# NYC TLC Gold Layer Analysis

Respuestas a las 20 preguntas usando las tablas `gold.*`.

Nombre: Matías Jaramillo
Clase: Data Mining
Tarea: PSet 02

In [1]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

In [5]:
required_vars = [
    "POSTGRES_HOST",
    "POSTGRES_PORT",
    "POSTGRES_USER",
    "POSTGRES_PASSWORD",
    "POSTGRES_DB",
]

missing = [k for k in required_vars if not os.getenv(k)]

if missing:
    env_path = Path(".env")
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            if key in required_vars and not os.getenv(key):
                os.environ[key] = value

missing = [k for k in required_vars if not os.getenv(k)]
if missing:
    raise ValueError(f"Missing required environment variables: {missing}")

DB_HOST = "localhost"
DB_PORT = os.getenv("POSTGRES_PORT")
DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DB")

print("Environment variables loaded successfully.")
print(f"POSTGRES_HOST={DB_HOST}")
print(f"POSTGRES_PORT={DB_PORT}")
print(f"POSTGRES_USER={DB_USER}")
print(f"POSTGRES_DB={DB_NAME}")

Environment variables loaded successfully.
POSTGRES_HOST=localhost
POSTGRES_PORT=5432
POSTGRES_USER=taxi_user
POSTGRES_DB=taxi_db


In [6]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def run_query(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, engine)

In [7]:
run_query("""
SELECT 
    (SELECT COUNT(*) FROM gold.fct_trips) AS fct_rows,
    (SELECT COUNT(*) FROM gold.dim_zone) AS dim_zone_rows,
    (SELECT COUNT(*) FROM gold.dim_date) AS dim_date_rows,
    (SELECT COUNT(*) FROM gold.dim_service_type) AS dim_service_type_rows,
    (SELECT COUNT(*) FROM gold.dim_payment_type) AS dim_payment_type_rows,
    (SELECT COUNT(*) FROM gold.dim_vendor) AS dim_vendor_rows
""")

,fct_rows,dim_zone_rows,dim_date_rows,dim_service_type_rows,dim_payment_type_rows,dim_vendor_rows
0,162781997,265,1453,2,3,5


## 1. Trips por mes (2024)

**Tablas usadas:** `gold.fct_trips`

In [8]:
q1 = """
SELECT
    date_trunc('month', pickup_date)::date AS month,
    COUNT(*) AS trips
FROM gold.fct_trips
WHERE pickup_date >= DATE '2024-01-01'
  AND pickup_date < DATE '2025-01-01'
GROUP BY 1
ORDER BY 1;
"""
df1 = run_query(q1)
df1

,month,trips
0,2024-01-01,2966022
1,2024-02-01,3006887
2,2024-03-01,3573690
3,2024-04-01,3505040
4,2024-05-01,3712417
5,2024-06-01,3522221
6,2024-07-01,3059612
7,2024-08-01,2961613
8,2024-09-01,3613158
9,2024-10-01,3803439


**Interpretacion:** Se observa que en el año 2024 existen épocas de alta demanda en taxi trips. Estos meses son marzo-junio y luego regresa la temporada alta en septiembre hasta el fin de año.

## 2. Trips por service type y month

**Tablas usadas:** `gold.fct_trips`

In [9]:
q2 = """
SELECT
    date_trunc('month', pickup_date)::date AS month,
    service_type_key,
    COUNT(*) AS trips
FROM gold.fct_trips
WHERE pickup_date >= DATE '2024-01-01'
  AND pickup_date < DATE '2025-01-01'
GROUP BY 1, 2
ORDER BY 1, 2;
"""
df2 = run_query(q2)
df2

,month,service_type_key,trips
0,2024-01-01,green,55931
1,2024-01-01,yellow,2910091
2,2024-02-01,green,52988
3,2024-02-01,yellow,2953899
4,2024-03-01,green,56762
5,2024-03-01,yellow,3516928
6,2024-04-01,green,55835
7,2024-04-01,yellow,3449205
8,2024-05-01,green,60288
9,2024-05-01,yellow,3652129


**Interpretacion:** Se observa claramente que el tipo de trip con "yellow" service type contibuye en su mayoria al volumen de trips por mes. Lo cual indica que los trips "yellow" tienen mucha mas demanda o mucha mas oferta que los trips "green".

## 3. Top 10 pickup zones (total 2024)

**Tablas usadas:** `gold.fct_trips`, `gold.dim_zone`

In [10]:
q3 = """
SELECT
    z.zone,
    COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone z
  ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date >= DATE '2024-01-01'
  AND f.pickup_date < DATE '2025-01-01'
GROUP BY z.zone
ORDER BY trips DESC
LIMIT 10;
"""
df3 = run_query(q3)
df3

,zone,trips
0,JFK Airport,1906407
1,Upper East Side South,1886927
2,Midtown Center,1881518
3,Upper East Side North,1713109
4,Midtown East,1395526
5,Times Sq/Theatre District,1358260
6,Penn Station/Madison Sq West,1336366
7,Lincoln Square East,1295691
8,LaGuardia Airport,1272149
9,Murray Hill,1157153


**Interpretacion:** Se pueden ver las top 10 zonas de pickup del 2024. Siendo la más popular JFK Airport y la menos popular del top 10 Murray Hill.

## 4. Top 10 dropoff zones (total 2024)

**Tablas usadas:** `gold.fct_trips`, `gold.dim_zone`

In [11]:
q4 = """
SELECT
    z.zone,
    COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone z
  ON f.do_zone_key = z.zone_key
WHERE f.pickup_date >= DATE '2024-01-01'
  AND f.pickup_date < DATE '2025-01-01'
GROUP BY z.zone
ORDER BY trips DESC
LIMIT 10;
"""
df4 = run_query(q4)
df4

,zone,trips
0,Upper East Side North,1805768
1,Upper East Side South,1712820
2,Midtown Center,1519454
3,Times Sq/Theatre District,1284610
4,Murray Hill,1198013
5,Midtown East,1169673
6,Lincoln Square East,1135883
7,Upper West Side South,1134225
8,East Chelsea,1062171
9,Lenox Hill West,1056920


**Interpretacion:** Aquí se puede ver el top 10 de los dropoff zones del 2024. La más popular siendo Upper East Side North y la menos popular del top 10 siendo Lenox Hill West.

## 5. Top 5 boroughs por month (pickup)

**Tablas usadas:** `gold.fct_trips`, `gold.dim_zone`

In [12]:
q5 = """
WITH ranked AS (
    SELECT
        date_trunc('month', f.pickup_date)::date AS month,
        z.borough,
        COUNT(*) AS trips,
        ROW_NUMBER() OVER (
            PARTITION BY date_trunc('month', f.pickup_date)::date
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM gold.fct_trips f
    JOIN gold.dim_zone z
      ON f.pu_zone_key = z.zone_key
    WHERE f.pickup_date >= DATE '2024-01-01'
      AND f.pickup_date < DATE '2025-01-01'
    GROUP BY 1, 2
)
SELECT month, borough, trips
FROM ranked
WHERE rn <= 5
ORDER BY month, trips DESC;
"""
df5 = run_query(q5)
df5

,month,borough,trips
0,2024-01-01,Manhattan,2643139
1,2024-01-01,Queens,280753
2,2024-01-01,Brooklyn,32575
3,2024-01-01,Bronx,7690
4,2024-01-01,NaN,1517
5,2024-02-01,Manhattan,2705433
6,2024-02-01,Queens,256948
7,2024-02-01,Brooklyn,35194
8,2024-02-01,Bronx,7610
9,2024-02-01,NaN,1378


**Interpretacion:** Se observan los top borough pickups por mes. Esto nos ayuda a identificar tendencias geograficas. En este caso se puede ver que siempre la mayoria de pickups se realizan en Manhattan, seguido por Queens, luego Brooklyn, Bronx y finalmente los boroughs no definidos. Este patron se repite en todos los meses del 2024.

## 6. Horas pico (top 5 horas) para cada dia de la semana

**Tablas usadas:** `gold.fct_trips`, `gold.dim_date`

In [13]:
q6 = """
WITH hourly AS (
    SELECT
        d.day_name,
        EXTRACT(HOUR FROM f.pickup_ts)::int AS pickup_hour,
        COUNT(*) AS trips,
        ROW_NUMBER() OVER (
            PARTITION BY d.day_name
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM gold.fct_trips f
    JOIN gold.dim_date d
      ON f.pickup_date_key = d.date_key
    GROUP BY 1, 2
)
SELECT day_name, pickup_hour, trips
FROM hourly
WHERE rn <= 5
ORDER BY day_name, trips DESC;
"""
df6 = run_query(q6)
df6

,day_name,pickup_hour,trips
0,Friday,18,1778470
1,Friday,17,1649347
2,Friday,19,1640122
3,Friday,15,1487237
4,Friday,16,1478700
5,Monday,18,1490014
6,Monday,17,1461493
7,Monday,15,1337370
8,Monday,16,1328903
9,Monday,14,1279533


**Interpretacion:** Se observa las horas más ocupadas para taxi trips. Se puede identificar una tendencia que todos los dias exceptuando domingo, la hora mas en demanda son las 5PM, la cual tiene sentido debido a que es la hora cuando muchas personas salen del trabajo.

## 7. Distribucion de viajes por dia de la semana (ranking)

**Tablas usadas:** `gold.fct_trips`, `gold.dim_date`

In [14]:
q7 = """
SELECT
    d.day_name,
    COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_date d
  ON f.pickup_date_key = d.date_key
GROUP BY d.day_name
ORDER BY trips DESC;
"""
df7 = run_query(q7)
df7

,day_name,trips
0,Thursday,25350794
1,Wednesday,24469328
2,Friday,24411970
3,Saturday,24395224
4,Tuesday,23222066
5,Sunday,20688650
6,Monday,20243965


**Interpretacion:** Ranking de los dias más transitados. Donde los dias jueves son los dias con más demanda y los dias lunes los que tienen menos demanda.

## 8. Ingresos total por mes

**Tablas usadas:** `gold.fct_trips`

In [15]:
q8 = """
SELECT
    date_trunc('month', pickup_date)::date AS month,
    SUM(total_amount) AS total_revenue
FROM gold.fct_trips
GROUP BY 1
ORDER BY 1;
"""
df8 = run_query(q8)
df8

,month,total_revenue
0,2001-01-01,3.541500e+02
1,2001-08-01,2.455000e+01
2,2002-10-01,1.141846e+04
3,2002-12-01,1.011180e+03
4,2003-01-01,6.052100e+02
5,2007-12-01,2.275000e+01
6,2008-12-01,3.342790e+03
7,2009-01-01,2.287130e+03
8,2012-02-01,0.000000e+00
9,2014-11-01,5.238000e+01


**Interpretacion:** Se pueden observar tendencias en los ingresos a traves de los años. Se relaciona con la anterior pregunta de numero de trips por mes, porque se pueden observar tendencias similares como las épocas de mayor demanda.

## 9. Ingreso total por service type y por mes

**Tablas usadas:** `gold.fct_trips`

In [16]:
q9 = """
SELECT
    date_trunc('month', pickup_date)::date AS month,
    service_type_key,
    SUM(total_amount) AS total_revenue
FROM gold.fct_trips
GROUP BY 1, 2
ORDER BY 1, 2;
"""
df9 = run_query(q9)
df9

,month,service_type_key,total_revenue
0,2001-01-01,yellow,3.541500e+02
1,2001-08-01,yellow,2.455000e+01
2,2002-10-01,yellow,1.141846e+04
3,2002-12-01,yellow,1.011180e+03
4,2003-01-01,yellow,6.052100e+02
...,...,...,...
105,2025-10-01,yellow,1.213315e+08
106,2025-11-01,green,1.169418e+06
107,2025-11-01,yellow,1.085070e+08
108,2025-12-01,green,2.943500e+02


**Interpretacion:** Similar a la pregunta anterior, nos indica tendencias en el tiempo sobre la demanda por cada service type, solo que tambien nos indica la diferencia entre los ingresos de los trips "yellow" y los trips "green".

## 10. Tip promedio por mes

**Tablas usadas:** `gold.fct_trips`

In [17]:
q10 = """
SELECT
    date_trunc('month', pickup_date)::date AS month,
    AVG(tip_amount / NULLIF(fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips
GROUP BY 1
ORDER BY 1;
"""
df10 = run_query(q10)
df10

,month,avg_tip_pct
0,2001-01-01,0.021665
1,2001-08-01,0.000000
2,2002-10-01,0.076697
3,2002-12-01,0.111385
4,2003-01-01,0.059114
5,2007-12-01,0.000000
6,2008-12-01,0.147055
7,2009-01-01,0.125709
8,2012-02-01,NaN
9,2014-11-01,0.263746


**Interpretacion:** Se muestra el porcentaje del fare que se recibe como tip. Se pueden observar las tendencias de tip a lo largo del tiempo.Se puede observar que el tip no cambia mucho, unicamente se puede ver que antes era mas en un 20% y en tiempos más recientes ha bajado a un 19% - 18%.

## 11. Tip % bpory borough y mes

**Tablas usadas:** `gold.fct_trips`, `gold.dim_zone`

In [18]:
q11 = """
SELECT
    date_trunc('month', f.pickup_date)::date AS month,
    z.borough,
    AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_zone z
  ON f.pu_zone_key = z.zone_key
GROUP BY 1, 2
ORDER BY 1, 2;
"""
df11 = run_query(q11)
df11

,month,borough,avg_tip_pct
0,2001-01-01,Manhattan,0.027855
1,2001-01-01,Queens,0.000000
2,2001-08-01,Queens,0.000000
3,2002-10-01,Bronx,0.000000
4,2002-10-01,Brooklyn,0.008242
...,...,...,...
351,2025-11-01,Queens,0.139789
352,2025-11-01,Staten Island,0.080210
353,2025-11-01,NaN,0.165480
354,2025-12-01,Manhattan,0.187601


**Interpretacion:** Aqui se puede observar la tendencia del tip a lo largo del tiempo y categorizado por borough. De esta manera se puede identificar flujos en tips  y el efecto que tiene el borough en la variable igualmente.

## 12. Top 10 pickup zones por ingreso total.

**Tablas usadas:** `gold.fct_trips`, `gold.dim_zone`

In [19]:
q12 = """
SELECT
    z.zone,
    SUM(f.total_amount) AS total_revenue
FROM gold.fct_trips f
JOIN gold.dim_zone z
  ON f.pu_zone_key = z.zone_key
GROUP BY z.zone
ORDER BY total_revenue DESC
LIMIT 10;
"""
df12 = run_query(q12)
df12

,zone,total_revenue
0,JFK Airport,5.742785e+08
1,LaGuardia Airport,3.038308e+08
2,Midtown Center,1.692282e+08
3,Upper East Side South,1.450595e+08
4,Times Sq/Theatre District,1.398398e+08
5,Upper East Side North,1.328292e+08
6,Penn Station/Madison Sq West,1.260662e+08
7,Midtown East,1.240145e+08
8,Lincoln Square East,1.063228e+08
9,Midtown North,1.057795e+08


**Interpretacion:** Se observan las top 10 pickup zones rankeadas por ingreso total. De esta manera se puede identificar que los trips más valiosos son los que tienen pickup en JFK Airport.

## 13. Top 10 pickup zones por tip % con minimo N trips

**Tablas usadas:** `gold.fct_trips`, `gold.dim_zone`

**N:** 1,000 trips

In [20]:
q13 = """
SELECT
    z.zone,
    COUNT(*) AS trips,
    AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_zone z
  ON f.pu_zone_key = z.zone_key
GROUP BY z.zone
HAVING COUNT(*) >= 1000
ORDER BY avg_tip_pct DESC
LIMIT 10;
"""
df13 = run_query(q13)
df13

,zone,trips,avg_tip_pct
0,Outside of NYC,140776,5.758538
1,Newark Airport,24434,4.099271
2,Cambria Heights,8987,1.521469
3,Bay Terrace/Fort Totten,3349,1.075968
4,Baisley Park,62768,0.829308
5,Astoria Park,1281,0.734793
6,Spuyten Duyvil/Kingsbridge,14891,0.680902
7,Belmont,9491,0.654706
8,East New York,83440,0.609023
9,Marine Park/Floyd Bennett Field,2259,0.548981


**Interpretacion:** Usar un minimo N, nos ayuda a no listar zonas con pocos viajes y altos tips, para excluir datos atípicos. Se puede ver que la zona con tips consistentes mas altos es Outside of NYC.

## 14. Cash vs card: trips, ingreso, tip %

**Tablas usadas:** `gold.fct_trips`

In [21]:
q14 = """
SELECT
    payment_type_key,
    COUNT(*) AS trips,
    SUM(total_amount) AS total_revenue,
    AVG(tip_amount / NULLIF(fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips
WHERE payment_type_key IN ('cash', 'card')
GROUP BY payment_type_key
ORDER BY trips DESC;
"""
df14 = run_query(q14)
df14

,payment_type_key,trips,total_revenue,avg_tip_pct
0,card,119535232,3.342808e+09,0.271517
1,cash,24004207,5.478649e+08,0.000020


**Interpretacion:** Se compara los metodos de pago para identificar patrones en los pagos. Se puede ver que pagos con tarjeta son mushisimos más que con cash, igualmente generan mucho más ingreso y suelen dejar mayor porcentaje de tip que cuando el metodo de pago es en cash.

## 15. Duracion promedio de trip (minutos) por mes

**Tablas usadas:** `gold.fct_trips`

In [22]:
q15 = """
SELECT
    date_trunc('month', pickup_date)::date AS month,
    AVG(trip_duration_minutes) AS avg_duration_minutes
FROM gold.fct_trips
GROUP BY 1
ORDER BY 1;
"""
df15 = run_query(q15)
df15

,month,avg_duration_minutes
0,2001-01-01,3.691241e+02
1,2001-08-01,2.243333e+01
2,2002-10-01,3.101679e+06
3,2002-12-01,5.019576e+02
4,2003-01-01,3.277250e+02
5,2007-12-01,1.700000e+01
6,2008-12-01,7.410979e+02
7,2009-01-01,4.417496e+02
8,2012-02-01,1.432217e+03
9,2014-11-01,4.274833e+02


**Interpretacion:** Se puede observar patrones de duracion de trip. Lo cual nos puede indicar condiciones de trafico o eventos que pudieron alentar trips en diferentes meses.

## 16. Distancia promedio por mes

**Tablas usadas:** `gold.fct_trips`

In [23]:
q16 = """
SELECT
    date_trunc('month', pickup_date)::date AS month,
    AVG(trip_distance) AS avg_trip_distance
FROM gold.fct_trips
GROUP BY 1
ORDER BY 1;
"""
df16 = run_query(q16)
df16

,month,avg_trip_distance
0,2001-01-01,8.395556
1,2001-08-01,7.080000
2,2002-10-01,3.782980
3,2002-12-01,8.538182
4,2003-01-01,12.159167
5,2007-12-01,3.000000
6,2008-12-01,8.512083
7,2009-01-01,4.351829
8,2012-02-01,0.000000
9,2014-11-01,6.430000


**Interpretacion:** Se pueden observar las tendencias en la distancia de los trips por mes. Esto puede ayudar a comprender patrones sobre el comportamiento dde los clientes al usar un taxi. Se usan mayormente para viajes cortos o largos?

## 17. Velocidad promedio (mph) por borough y franja horaria

**Tablas usadas:** `gold.fct_trips`, `gold.dim_zone`

In [24]:
q17 = """
SELECT
    z.borough,
    CASE
        WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 6 AND 11 THEN 'morning'
        WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 12 AND 17 THEN 'afternoon'
        WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 18 AND 23 THEN 'evening'
        ELSE 'night'
    END AS time_band,
    AVG(f.trip_distance / NULLIF(f.trip_duration_minutes / 60.0, 0)) AS avg_mph
FROM gold.fct_trips f
JOIN gold.dim_zone z
  ON f.pu_zone_key = z.zone_key
GROUP BY 1, 2
ORDER BY 1, 2;
"""
df17 = run_query(q17)
df17

,borough,time_band,avg_mph
0,Bronx,afternoon,180.152577
1,Bronx,evening,168.140578
2,Bronx,morning,201.299619
3,Bronx,night,94.409876
4,Brooklyn,afternoon,90.880735
5,Brooklyn,evening,70.642798
6,Brooklyn,morning,144.939079
7,Brooklyn,night,80.414735
8,EWR,afternoon,67.767989
9,EWR,evening,60.300375


**Interpretacion:** Se puede observar la velocidad promedio del trip con su respectivo borough y franja horaria. Con esta data se puede aprender sobre los patrones de congestion en diferentes fanjas horarias y categorizadas por borough, lo cual puede luego ayudar a opeaciones logisticas para cumplir con la demanda.

## 18. Percentiles p50 y p90 de duracion por borough

**Tablas usadas:** `gold.fct_trips`, `gold.dim_zone`

In [25]:
q18 = """
SELECT
    z.borough,
    percentile_cont(0.5) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p50_duration,
    percentile_cont(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p90_duration
FROM gold.fct_trips f
JOIN gold.dim_zone z
  ON f.pu_zone_key = z.zone_key
GROUP BY z.borough
ORDER BY z.borough;
"""
df18 = run_query(q18)
df18

,borough,p50_duration,p90_duration
0,Bronx,23.183333,61.483333
1,Brooklyn,21.533333,48.550000
2,EWR,0.150000,1.366667
3,Manhattan,11.883333,26.300000
4,Queens,31.916667,60.766667
5,Staten Island,29.450000,72.820000
6,NaN,0.800000,60.266667


**Interpretacion:** p50 indica duracion normal mientras que p90 indica duraciones largas. De esta manera se puede comparar lo "normal" y lo "demorado" categorizado por borough.

## 19. Top 10 pickup zones por duracion p90

**Tablas usadas:** `gold.fct_trips`, `gold.dim_zone`

In [26]:
q19 = """
SELECT
    z.zone,
    percentile_cont(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p90_duration
FROM gold.fct_trips f
JOIN gold.dim_zone z
  ON f.pu_zone_key = z.zone_key
GROUP BY z.zone
ORDER BY p90_duration DESC
LIMIT 10;
"""
df19 = run_query(q19)
df19

,zone,p90_duration
0,Arden Heights,124.086667
1,Freshkills Park,102.086667
2,Far Rockaway,100.010000
3,Hammels/Arverne,97.335000
4,Coney Island,92.698333
5,Eltingville/Annadale/Prince's Bay,90.876667
6,Rockaway Park,88.230000
7,Brighton Beach,84.110000
8,Gravesend,83.266667
9,Cambria Heights,80.893333


**Interpretacion:** Se observan las zonas con el p90 mas alto (top 10). Son indicadores de los viajes mas largos. Esto puede indicar zonas donde es mas probable que haya congestion o demoras.

## 20. Top 10 borough-to-borough rutas por trip count

**Tablas usadas:** `gold.fct_trips`, `gold.dim_zone`

In [27]:
q20 = """
SELECT
    pu.borough AS pickup_borough,
    do_.borough AS dropoff_borough,
    COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone pu
  ON f.pu_zone_key = pu.zone_key
JOIN gold.dim_zone do_
  ON f.do_zone_key = do_.zone_key
GROUP BY pu.borough, do_.borough
ORDER BY trips DESC
LIMIT 10;
"""
df20 = run_query(q20)
df20

,pickup_borough,dropoff_borough,trips
0,Manhattan,Manhattan,133950293
1,Queens,Manhattan,8271652
2,Manhattan,Queens,4707004
3,Queens,Queens,4283676
4,Manhattan,Brooklyn,3369054
5,Queens,Brooklyn,2265680
6,Brooklyn,Brooklyn,1754410
7,Brooklyn,Manhattan,971431
8,Manhattan,Bronx,638249
9,Queens,NaN,402625


**Interpretacion:** Se muestran los top 10 viajes entre boroughs. Esto nos puede dar evidencias de patrones y flujo de movimiento en la ciudad. Por ejemplo, en este caso se observa que la ruta mas popular es movimientos que se mantienen dentro de Manhattan.